In [14]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

In [15]:
df = pd.read_csv("EMS_Feature_Engineered.csv")

print(df.shape)

df.head()

(3629661, 29)


,initial_call_type,final_call_type,dispatch_response_seconds_qy,incident_response_seconds_qy,incident_travel_tm_seconds_qy,zipcode,borough,incident_dispatch_area,hour,month,...,closure_time,zipcode_frequency,borough_frequency,dispatch_area_frequency,initial_call_frequency,final_call_frequency,rush_hour,office_hours,call_borough,zip_hour
0,53.0,51.0,53,393.0,340.0,11365.0,3.0,26.0,0.0,1.0,...,-8276829.0,9311,743558,43405,437223,416373,0,0,337,4244
1,21.0,17.0,84,430.0,346.0,10457.0,0.0,1.0,0.0,1.0,...,3195.0,59102,854212,253459,188550,197297,0,0,180,2298
2,99.0,111.0,0,564.0,433.0,11224.0,1.0,6.0,0.0,1.0,...,6840.0,34013,1002290,150556,985,8795,0,0,536,3547
3,84.0,95.0,1485,1934.0,449.0,10033.0,2.0,21.0,0.0,1.0,...,2783.0,25033,873477,98891,33155,27167,0,0,470,744
4,20.0,16.0,39,1315.0,1276.0,11208.0,1.0,9.0,0.0,1.0,...,4144.0,51519,1002290,252294,3778,3513,0,0,176,3163


In [41]:
df["eta_seconds"] = (
    df["dispatch_response_seconds_qy"] +
    df["incident_travel_tm_seconds_qy"]
)

In [42]:
df = df.dropna(subset=["eta_seconds"])

In [43]:
features = [

    "initial_call_type",

    "final_call_type",

    "zipcode",

    "borough",

    "incident_dispatch_area",

    "hour",

    "month",

    "weekday",

    "Weekend",

    "Year",

    "incident_response_seconds_qy",

    "dispatch_delay",

    "zipcode_frequency",

    "borough_frequency",

    "dispatch_area_frequency",

    "initial_call_frequency",

    "final_call_frequency",

    "call_borough",

    "zip_hour"

]

In [44]:
target = "eta_seconds"

In [45]:
X = df[features]

y = df[target]

In [46]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42

)

In [49]:
models = {

    "XGBoost":

        XGBRegressor(

            n_estimators=300,

            learning_rate=0.05,

            max_depth=8,

            subsample=0.8,

            colsample_bytree=0.8,

            objective="reg:squarederror",

            tree_method="hist",

            random_state=42,

            n_jobs=-1

        ),

    "CatBoost":

        CatBoostRegressor(

            iterations=300,

            learning_rate=0.05,

            depth=8,

            verbose=100,

            random_seed=42

        )

}

In [50]:
results = []

trained_models = {}

for name, model in models.items():

    print("="*60)

    print(name)

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    pred = np.maximum(pred,0)

    mae = mean_absolute_error(y_test,pred)

    rmse = np.sqrt(mean_squared_error(y_test,pred))

    r2 = r2_score(y_test,pred)

    print("MAE :",round(mae,2))

    print("RMSE:",round(rmse,2))

    print("R2 :",round(r2,4))

    results.append([

        name,

        mae,

        rmse,

        r2

    ])

    trained_models[name]=model

XGBoost
MAE : 44.45
RMSE: 369.24
R2 : 0.8969
CatBoost
0:	learn: 1108.1116980	total: 556ms	remaining: 2m 46s
100:	learn: 394.1545318	total: 56.3s	remaining: 1m 51s
200:	learn: 382.5356763	total: 1m 55s	remaining: 57s
299:	learn: 375.8332417	total: 2m 53s	remaining: 0us
MAE : 45.67
RMSE: 376.19
R2 : 0.893


In [53]:
results_df = pd.DataFrame(

    results,

    columns=[

        "Model",

        "MAE",

        "RMSE",

        "R2 Score"

    ]

)

results_df.sort_values(

    by="R2 Score",

    ascending=False

)

,Model,MAE,RMSE,R2 Score
0,XGBoost,44.454249,369.242307,0.896941
1,CatBoost,45.668570,376.189647,0.893027


In [54]:
best_model_name = results_df.iloc[0]["Model"]

best_model = trained_models[best_model_name]

print(best_model_name)

XGBoost


In [55]:
sample = X_test.iloc[:5]

prediction = best_model.predict(sample)

prediction

array([577.12256, 223.82289, 401.56058, 927.52716,  53.48836],
      dtype=float32)

In [56]:
import joblib

joblib.dump(

    best_model,

    "ETA_Prediction_Model.pkl"

)

['ETA_Prediction_Model.pkl']

In [57]:
from google.colab import files

files.download("ETA_Prediction_Model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>